# MySQL 写入 DolphinDB 指南

这个 notebook 结合当前仓库里的实现，说明如何把 MySQL 数据写入 DolphinDB。重点是三层能力：

1. **查看与理解配置**：确认 MySQL、DolphinDB、checkpoint、job 配置。
2. **手动跑一批数据**：用底层 reader / transformer / writer 验证链路。
3. **运行完整 ETL**：直接调用 `ETLPipeline` 或 CLI 执行作业。

当前仓库里的主流程是：

- `etl/readers/mysql_chunk_reader.py`：分批从 MySQL 读取
- `etl/transformers/base.py`：字段改名和类型转换
- `etl/writers/dolphindb_writer.py`：写入 DolphinDB
- `etl/pipeline.py`：把读取、转换、写入、checkpoint 串起来

> 下面的示例默认**不会自动执行写入**，只有你主动调用运行函数时才会真的写 DolphinDB。


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import yaml

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from etl.checkpoint import CheckpointManager
from etl.pipeline import ETLPipeline
from etl.readers.mysql_chunk_reader import MySQLChunkReader
from etl.transformers.base import DataTransformer
from etl.writers.dolphindb_writer import DolphinDBWriter
from etl.writers.strategies.factory import build_write_strategy

CONFIG_PATH = REPO_ROOT / 'config' / 'config.yaml'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'CONFIG_PATH = {CONFIG_PATH}')


## 1. 查看当前配置

项目的 ETL 入口依赖 `config/config.yaml`。里面最关键的是：

- `mysql`：源库连接
- `dolphindb`：目标库连接
- `checkpoint`：断点续传保存位置
- `jobs`：每张表如何抽取、映射、写入

当前实现支持从 keyring 读取密码，所以 notebook 不需要把明文密码写死。


In [ ]:
pd.DataFrame(CONFIG['jobs'])[['name', 'mysql_table', 'dolphindb_table', 'incremental_col', 'writer_strategy', 'chunk_size']]


In [ ]:
job_name = CONFIG['jobs'][0]['name']
job = next(j for j in CONFIG['jobs'] if j['name'] == job_name)
job


## 2. 一条链路里到底发生了什么

以一个 job 为例，完整流程如下：

1. `CheckpointManager.load(job_name)` 读取上次同步到哪里
2. `MySQLChunkReader.read_batches()` 按 `incremental_col` 从 MySQL 分批读取
3. `DataTransformer.transform()` 按 `field_mapping` 重命名字段并转换类型
4. `DolphinDBWriter.write_batch()` 按写入策略写入 DolphinDB
5. 每批成功后保存 checkpoint，失败时下次可续跑


In [ ]:
checkpoint_mgr = CheckpointManager(CONFIG['checkpoint']['db_path'])
checkpoint = checkpoint_mgr.load(job['name'])
checkpoint


In [ ]:
reader = MySQLChunkReader(CONFIG['mysql'], job, checkpoint)
transformer = DataTransformer(job['field_mapping'])
strategy = build_write_strategy(job)

print(type(reader).__name__)
print(type(transformer).__name__)
print(type(strategy).__name__)


## 3. 先只读取一批 MySQL 数据

这一步适合先验证：

- 连接是否正常
- where 条件是否符合预期
- 增量列是否能正确推进
- 字段内容是否适合后续写入


In [ ]:
batch_iter = reader.read_batches()
first_df, first_batch_id = next(batch_iter)
print('batch_id =', first_batch_id)
print('rows =', len(first_df))
first_df.head()


## 4. 做字段映射和类型转换

这里会把 MySQL 字段名转换成 DolphinDB 目标字段名，并按 `field_mapping` 里的类型做处理。

例如：

- `SecuCode -> secuCode`
- `ListedDate -> listedDate`
- `TIMESTAMP` 会转成 pandas datetime


In [ ]:
transformed_df = transformer.transform(first_df.copy())
print(transformed_df.dtypes)
transformed_df.head()


## 5. 手动写入 DolphinDB

确认读取和转换没问题后，可以手动把这一批写入 DolphinDB。

当前 job 使用的写入策略来自 `writer_strategy`：

- `window_overwrite`：按时间窗口删除后重写
- `append_only`：只追加
- `key_delete_insert`：按主键/业务键删除后插入

默认先不要执行下面最后一行；确认目标表无误后再取消注释。


In [ ]:
writer = DolphinDBWriter(CONFIG['dolphindb'], job['dolphindb_table'], strategy)
print('target_db =', writer.db_path)
print('target_table =', writer.table_name)

# 真正写入时执行下面一行
# writer.write_batch(transformed_df, first_batch_id)


## 6. 用完整 Pipeline 跑一个 job

如果你不想手动分步执行，最直接的方法就是调用项目里已经封装好的 `ETLPipeline`。

它会自动完成：

- 读取 checkpoint
- 分批拉取 MySQL
- 转换
- 写入 DolphinDB
- 更新 checkpoint


In [ ]:
def run_pipeline(job_name: str):
    job_conf = next(j for j in CONFIG['jobs'] if j['name'] == job_name)
    checkpoint_mgr = CheckpointManager(CONFIG['checkpoint']['db_path'])
    pipeline = ETLPipeline(job_conf, CONFIG['mysql'], CONFIG['dolphindb'], checkpoint_mgr)
    pipeline.run()
    return checkpoint_mgr.load(job_name)

# 示例：真正执行时取消注释
# run_pipeline(job_name)


## 7. 命令行方式

除了 notebook，仓库本身也已经提供了 CLI 入口。你可以在项目根目录执行：

```bash
python -m etl --list
python -m etl -j secumain_sync
```

其中：

- `--list` 列出所有 job
- `-j <job_name>` 只执行一个 job

这和 notebook 里的 `run_pipeline(job_name)` 本质上是同一套实现。


## 8. 新增一张 MySQL 表时怎么接入

如果你要把新的 MySQL 表写入 DolphinDB，通常按这个顺序做：

1. 在 MySQL 确认源表名、增量列、过滤条件。
2. 在 DolphinDB 先建好目标库表。
3. 在 `config/config.yaml` 里新增一个 job：
   - `mysql_table`
   - `dolphindb_table`
   - `incremental_col`
   - `time_column`
   - `writer_strategy`
   - `field_mapping`
4. 先在 notebook 里跑一批验证。
5. 再跑完整 pipeline。

仓库里还提供了自动生成映射的脚本：

```bash
python scripts/generate_table_mapping.py <table_name>
```

它会帮你生成 `jobs` 配置片段、DolphinDB 建表脚本和 inventory 信息。


## 9. 推荐的实际操作顺序

建议你每次接入新表时按下面顺序操作：

1. 先看 job 配置是否正确。
2. 先执行“只读取一批 MySQL”。
3. 再执行“转换后预览”。
4. 手动写入一批到测试表。
5. 最后运行完整 ETL。

这样最容易排查问题，尤其是字段类型、时间列、写入策略和 checkpoint。


In [ ]:
reader.close()
writer.close()
